# 0.4 - Predictive Model On Claim Status

Uses cleaned but cleaning logic included if cleaned files are not prepared

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, mean_absolute_error,
    mean_squared_error, r2_score, roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

support_file = Path("SupportData.xlsx")
gl_account_file = Path("GLAccountData.xlsx")
gl_entry_file = Path("GLEntryData.xlsx")

clean_data_dir = Path("outputs") / "cleaned_data"
support_cleaned_file = clean_data_dir / "support_cleaned.csv"
gl_accounts_cleaned_file = clean_data_dir / "gl_accounts_cleaned.csv"
gl_entries_cleaned_file = clean_data_dir / "gl_entries_cleaned.csv"

def read_cleaned_dataset(path):
    if not path.exists():
        raise FileNotFoundError(f"Cleaned dataset not found: {path}. Run 0.2 - Data Cleaning first.")
    df = pd.read_csv(path, low_memory=False)
    for col in df.columns:
        if "date" in col.lower():
            df[col] = pd.to_datetime(df[col], errors="coerce")
    return df

random_state = 42
analysis_start = pd.Timestamp("2026-01-01")
analysis_support_end = pd.Timestamp("2026-05-31")
analysis_gl_end = pd.Timestamp("2026-06-30")

def clean_columns(df):
    df = df.copy()
    df.columns = df.columns.astype(str).str.strip().str.replace(".", "_", regex=False).str.replace(" ", "_", regex=False)
    return df

def normalise_key(series):
    return series.astype("string").str.strip().str.upper().str.replace(r"\.0$", "", regex=True)

def normalise_gl_description(series):
    return (series.astype("string").str.upper()
        .str.replace(r"\bCLAIMBACKS?\b", "", regex=True)
        .str.replace(r"\b[A-Z]\d+[A-Z]*\b", "", regex=True)
        .str.replace(r"\bLIMITED\b|\bLTD\b", "", regex=True)
        .str.replace(r"\s+", " ", regex=True).str.strip())

def build_model_base_from_support():
    df = clean_columns(read_cleaned_dataset(support_cleaned_file))
    df["claim_line_id"] = np.arange(1, len(df) + 1)
    df["claim_date"] = pd.to_datetime(df.get("Date"), errors="coerce", dayfirst=True)
    df["customer_date"] = df["claim_date"]
    df["customer_year"] = df["claim_date"].dt.year
    df["customer_month"] = df["claim_date"].dt.month
    qty = pd.to_numeric(df.get("SalesDeliveryNoteLine_Quantity"), errors="coerce").fillna(pd.to_numeric(df.get("SalesInvoiceLine_Quantity"), errors="coerce")).fillna(0)
    unit = pd.to_numeric(df.get("UnitClaimAmount"), errors="coerce").fillna(0)
    total = pd.to_numeric(df.get("TotalClaimAmount"), errors="coerce")
    df["credit_due_value"] = total.fillna(unit * qty)
    credit_note = df.get("SalesCreditNote_Number", pd.Series("", index=df.index)).astype("string").str.strip().ne("").fillna(False)
    df["credit_received_value"] = np.where(credit_note.to_numpy(dtype=bool), df["credit_due_value"].clip(lower=0), 0)
    df["outstanding_credit_value"] = df["credit_due_value"] - df["credit_received_value"]
    df["recovered_credit_rate"] = df["credit_received_value"].div(df["credit_due_value"].replace(0, np.nan)).fillna(0)
    df["supplier_reference_value"] = df["credit_due_value"]
    df["supplier_reference_match_status"] = "Derived from SupportData.xlsx"
    has_received_credit = df["credit_received_value"].gt(0).fillna(False)
    has_outstanding_credit = df["outstanding_credit_value"].gt(0).fillna(False)
    df["credit_recovery_status"] = np.select(
        [has_received_credit.to_numpy(dtype=bool), has_outstanding_credit.to_numpy(dtype=bool)],
        ["Credit note present", "Outstanding"],
        default="No claim value",
    )
    df["days_outstanding"] = (pd.Timestamp.today().normalize() - df["claim_date"]).dt.days
    df["ageing_band"] = pd.cut(df["days_outstanding"], [-1,30,60,90,180,365,np.inf], labels=["0-30","31-60","61-90","91-180","181-365","365+"])
    df["requires_review"] = has_outstanding_credit
    df["review_reason"] = np.where(has_outstanding_credit.to_numpy(dtype=bool), "Outstanding credit value", "")
    return df

def load_gl_context():
    accounts = clean_columns(read_cleaned_dataset(gl_accounts_cleaned_file))
    entries = clean_columns(read_cleaned_dataset(gl_entries_cleaned_file))
    if not accounts.empty:
        accounts["gl_account_code"] = normalise_key(accounts.get("Code", pd.Series(pd.NA,index=accounts.index)))
        accounts["gl_account_description"] = accounts.get("Description", pd.Series(pd.NA,index=accounts.index))
        accounts["gl_supplier_key"] = normalise_gl_description(accounts["gl_account_description"])
        accounts["gl_current_balance"] = pd.to_numeric(accounts.get("CurrentBalance", pd.NA), errors="coerce")
        accounts["gl_current_balance_abs"] = accounts["gl_current_balance"].abs()
    if not entries.empty:
        entries["gl_account_code"] = normalise_key(entries.get("GL_Account_Code", pd.Series(pd.NA,index=entries.index)))
        entries["gl_account_description"] = entries.get("Description", pd.Series(pd.NA,index=entries.index))
        entries["gl_supplier_key"] = normalise_gl_description(entries["gl_account_description"])
        entries["gl_credit_amount"] = pd.to_numeric(entries.get("Credit_Amount", pd.NA), errors="coerce").fillna(0)
        entries["gl_entry_line_date"] = pd.to_datetime(entries.get("Entry_Line_Date", pd.NA), errors="coerce", dayfirst=True)
    if not entries.empty:
        entries["gl_entry_period_month"] = entries["gl_entry_line_date"].dt.to_period("M").astype("string")
        summary = entries.groupby(["gl_account_code","gl_account_description","gl_supplier_key"], dropna=False).agg(
            gl_entry_rows=("gl_account_code","size"),
            gl_total_credit_amount=("gl_credit_amount","sum"),
            gl_first_entry_date=("gl_entry_line_date","min"),
            gl_last_entry_date=("gl_entry_line_date","max"),
        ).reset_index()
        if not accounts.empty:
            account_cols = [col for col in ["gl_account_code", "gl_current_balance", "gl_current_balance_abs"] if col in accounts.columns]
            summary = summary.merge(accounts[account_cols].drop_duplicates("gl_account_code"), on="gl_account_code", how="left")
    else:
        summary = pd.DataFrame()
    return accounts, entries, summary

df = build_model_base_from_support()
gl_accounts_clean, gl_entries_clean, gl_account_summary_clean = load_gl_context()

print("Loaded cleaned datasets from outputs/cleaned_data.")
print("Model base shape:", df.shape)
print("GL summary shape:", gl_account_summary_clean.shape)

gl_accounts = gl_accounts_clean
gl_entries = gl_entries_clean
gl_account_summary = gl_account_summary_clean
supplier_gl_payment_model_base = pd.DataFrame()

if "supplier_group_code" not in gl_account_summary_clean.columns and "gl_supplier_key" in gl_account_summary_clean.columns:
    gl_account_summary_clean = gl_account_summary_clean.copy()
    gl_account_summary_clean["supplier_group_code"] = gl_account_summary_clean["gl_supplier_key"]
    gl_account_summary = gl_account_summary_clean

display(df.head())
if "Contract_Supplier_Code" in df.columns:
    supplier_group_code = dict(zip(df["Contract_Supplier_Code"], normalise_key(df["Contract_Supplier_Code"])))
else:
    supplier_group_code = {}

display(gl_account_summary_clean.head())

Loaded cleaned datasets from outputs/cleaned_data.
Model base shape: (73320, 57)
GL summary shape: (15, 9)


C:\Users\cathalhe\AppData\Local\Temp\ipykernel_31500\3719851810.py:35: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df[col] = pd.to_datetime(df[col], errors="coerce")


,Date,Month,Year,SalesDeliveryNote_Branch,Customer,Customer_Name,Product,Product_ManufacturerProductCode,Contract_Number,SourceTransactionType,Contract_D_ContractNumber,SalesInvoice_Number,SalesCreditNote_Number,SalesDeliveryNote_Number,SalesReturnNote_Number,SalesOrder_Number,Status,UnitClaimAmount,TotalClaimAmount,SalesDeliveryNoteLine_Quantity,SalesInvoiceLine_Quantity,SalesInvoice_Branch,SalesDeliveryNoteLine_DiscountPercentage,Product_Category,SalesCreditNoteLine_D_InvoiceCost,SalesCreditNoteLine_D_FixedCost,SalesDeliveryNoteLine_D_InvoiceCost,SalesDeliveryNoteLine_D_FixedCost,SalesInvoiceLine_D_InvoiceCost,SalesInvoiceLine_D_FixedCost,SalesDeliveryNote_Branch_1,SalesDeliveryNote_NetAmountLessDiscountBase,Product_CSQL_InvoiceCostForbranch,Product_CSQL_ListPriceForBranch,Product_CALC_FixedCostForBranch,Contract_Description,Contract_Expression,Contract_ValidFrom,Contract_ValidTo,Contract_Supplier_Code,Contract_Supplier_Name,claim_line_id,claim_date,customer_date,customer_year,customer_month,credit_due_value,credit_received_value,outstanding_credit_value,recovered_credit_rate,supplier_reference_value,supplier_reference_match_status,credit_recovery_status,days_outstanding,ageing_band,requires_review,review_reason
0,2024-02-26,NaN,NaN,2.0,M3M500,M3 Mechanical Limited,YXS1228,38300,86.0,SL/Del,039-C-0692,NaN,NaN,SO0000413/1,NaN,SO0000413,Claimed,1.366861,8.2012,6,0,NaN,69.0,PRSCU,0.0,0.0,5.4894,2.6953,0.0,0.0,2.0,392.18,6.235348,14.21,4.614158,M3 MECHANICAL - YX,[D_InvoiceCost]*0.2490,01/01/0001,01/01/0001,U106,Supplier1,1,2024-02-26,2024-02-26,2024,2,8.2012,0.0,8.2012,0.0,8.2012,Derived from SupportData.xlsx,Outstanding,928,365+,True,Outstanding credit value
1,2024-02-26,NaN,NaN,2.0,M3M500,M3 Mechanical Limited,YXS12S28,38322,86.0,SL/Del,039-C-0692,NaN,NaN,SO0000413/1,NaN,SO0000413,Claimed,1.783139,3.5663,2,0,NaN,69.0,PRSCU,0.0,0.0,7.1612,3.5162,0.0,0.0,2.0,392.18,8.130964,18.53,6.016913,M3 MECHANICAL - YX,[D_InvoiceCost]*0.2490,01/01/0001,01/01/0001,U106,Supplier1,2,2024-02-26,2024-02-26,2024,2,3.5663,0.0,3.5663,0.0,3.5663,Derived from SupportData.xlsx,Outstanding,928,365+,True,Outstanding credit value
2,2024-02-26,NaN,NaN,2.0,M3M500,M3 Mechanical Limited,YXS62822,38204,86.0,SL/Del,039-C-0692,NaN,NaN,SO0000413/1,NaN,SO0000413,Claimed,1.190942,2.3819,2,0,NaN,69.0,PRSCU,0.0,0.0,4.7829,2.3484,0.0,0.0,2.0,392.18,5.432344,12.38,4.019935,M3 MECHANICAL - YX,[D_InvoiceCost]*0.2490,01/01/0001,01/01/0001,U106,Supplier1,3,2024-02-26,2024-02-26,2024,2,2.3819,0.0,2.3819,0.0,2.3819,Derived from SupportData.xlsx,Outstanding,928,365+,True,Outstanding credit value
3,2024-02-26,NaN,NaN,2.0,M3M500,M3 Mechanical Limited,YXS252815,38492,86.0,SL/Del,039-C-0692,NaN,NaN,SO0000413/1,NaN,SO0000413,Claimed,2.488979,9.9559,4,0,NaN,69.0,PRSCU,0.0,0.0,9.9959,4.9080,0.0,0.0,2.0,392.18,11.351756,25.87,8.400299,M3 MECHANICAL - YX,[D_InvoiceCost]*0.2490,01/01/0001,01/01/0001,U106,Supplier1,4,2024-02-26,2024-02-26,2024,2,9.9559,0.0,9.9559,0.0,9.9559,Derived from SupportData.xlsx,Outstanding,928,365+,True,Outstanding credit value
4,2024-02-26,NaN,NaN,2.0,M3M500,M3 Mechanical Limited,YXS252215,38490,86.0,SL/Del,039-C-0692,NaN,NaN,SO0000413/1,NaN,SO0000413,Claimed,0.890474,5.3428,6,0,NaN,69.0,PRSCU,0.0,0.0,3.5762,1.7559,0.0,0.0,2.0,392.18,4.058900,9.25,3.003586,M3 MECHANICAL - YX,[D_InvoiceCost]*0.2490,01/01/0001,01/01/0001,U106,Supplier1,5,2024-02-26,2024-02-26,2024,2,5.3428,0.0,5.3428,0.0,5.3428,Derived from SupportData.xlsx,Outstanding,928,365+,True,Outstanding credit value


,gl_account_code,gl_account_description,gl_supplier_key,gl_entry_rows,gl_total_credit_amount,gl_first_entry_date,gl_last_entry_date,gl_current_balance,gl_current_balance_abs,supplier_group_code
0,01-85001,Supplier13 E071 Claimbacks,SUPPLIER13,105,242784.06,2026-05-01,2026-07-03,-242784.06,242784.06,SUPPLIER13
1,01-85002,Supplier2 E005 Claimbacks,SUPPLIER2,1,6240.00,2026-02-25,2026-02-25,-6240.00,6240.00,SUPPLIER2
2,01-85007,Supplier9 E022 Claimbacks,SUPPLIER9,5,5774.89,2026-02-09,2026-07-10,-5774.89,5774.89,SUPPLIER9
3,01-85017,Supplier3 E055 Claimbacks,SUPPLIER3,4,5550.00,2026-03-17,2026-07-14,-5550.00,5550.00,SUPPLIER3
4,01-85021,Supplier19 E070 Claimbacks,SUPPLIER19,4,9124.28,2026-04-20,2026-04-20,-9124.28,9124.28,SUPPLIER19


In [2]:
display(df.head())


,Date,Month,Year,SalesDeliveryNote_Branch,Customer,Customer_Name,Product,Product_ManufacturerProductCode,Contract_Number,SourceTransactionType,Contract_D_ContractNumber,SalesInvoice_Number,SalesCreditNote_Number,SalesDeliveryNote_Number,SalesReturnNote_Number,SalesOrder_Number,Status,UnitClaimAmount,TotalClaimAmount,SalesDeliveryNoteLine_Quantity,SalesInvoiceLine_Quantity,SalesInvoice_Branch,SalesDeliveryNoteLine_DiscountPercentage,Product_Category,SalesCreditNoteLine_D_InvoiceCost,SalesCreditNoteLine_D_FixedCost,SalesDeliveryNoteLine_D_InvoiceCost,SalesDeliveryNoteLine_D_FixedCost,SalesInvoiceLine_D_InvoiceCost,SalesInvoiceLine_D_FixedCost,SalesDeliveryNote_Branch_1,SalesDeliveryNote_NetAmountLessDiscountBase,Product_CSQL_InvoiceCostForbranch,Product_CSQL_ListPriceForBranch,Product_CALC_FixedCostForBranch,Contract_Description,Contract_Expression,Contract_ValidFrom,Contract_ValidTo,Contract_Supplier_Code,Contract_Supplier_Name,claim_line_id,claim_date,customer_date,customer_year,customer_month,credit_due_value,credit_received_value,outstanding_credit_value,recovered_credit_rate,supplier_reference_value,supplier_reference_match_status,credit_recovery_status,days_outstanding,ageing_band,requires_review,review_reason
0,2024-02-26,NaN,NaN,2.0,M3M500,M3 Mechanical Limited,YXS1228,38300,86.0,SL/Del,039-C-0692,NaN,NaN,SO0000413/1,NaN,SO0000413,Claimed,1.366861,8.2012,6,0,NaN,69.0,PRSCU,0.0,0.0,5.4894,2.6953,0.0,0.0,2.0,392.18,6.235348,14.21,4.614158,M3 MECHANICAL - YX,[D_InvoiceCost]*0.2490,01/01/0001,01/01/0001,U106,Supplier1,1,2024-02-26,2024-02-26,2024,2,8.2012,0.0,8.2012,0.0,8.2012,Derived from SupportData.xlsx,Outstanding,928,365+,True,Outstanding credit value
1,2024-02-26,NaN,NaN,2.0,M3M500,M3 Mechanical Limited,YXS12S28,38322,86.0,SL/Del,039-C-0692,NaN,NaN,SO0000413/1,NaN,SO0000413,Claimed,1.783139,3.5663,2,0,NaN,69.0,PRSCU,0.0,0.0,7.1612,3.5162,0.0,0.0,2.0,392.18,8.130964,18.53,6.016913,M3 MECHANICAL - YX,[D_InvoiceCost]*0.2490,01/01/0001,01/01/0001,U106,Supplier1,2,2024-02-26,2024-02-26,2024,2,3.5663,0.0,3.5663,0.0,3.5663,Derived from SupportData.xlsx,Outstanding,928,365+,True,Outstanding credit value
2,2024-02-26,NaN,NaN,2.0,M3M500,M3 Mechanical Limited,YXS62822,38204,86.0,SL/Del,039-C-0692,NaN,NaN,SO0000413/1,NaN,SO0000413,Claimed,1.190942,2.3819,2,0,NaN,69.0,PRSCU,0.0,0.0,4.7829,2.3484,0.0,0.0,2.0,392.18,5.432344,12.38,4.019935,M3 MECHANICAL - YX,[D_InvoiceCost]*0.2490,01/01/0001,01/01/0001,U106,Supplier1,3,2024-02-26,2024-02-26,2024,2,2.3819,0.0,2.3819,0.0,2.3819,Derived from SupportData.xlsx,Outstanding,928,365+,True,Outstanding credit value
3,2024-02-26,NaN,NaN,2.0,M3M500,M3 Mechanical Limited,YXS252815,38492,86.0,SL/Del,039-C-0692,NaN,NaN,SO0000413/1,NaN,SO0000413,Claimed,2.488979,9.9559,4,0,NaN,69.0,PRSCU,0.0,0.0,9.9959,4.9080,0.0,0.0,2.0,392.18,11.351756,25.87,8.400299,M3 MECHANICAL - YX,[D_InvoiceCost]*0.2490,01/01/0001,01/01/0001,U106,Supplier1,4,2024-02-26,2024-02-26,2024,2,9.9559,0.0,9.9559,0.0,9.9559,Derived from SupportData.xlsx,Outstanding,928,365+,True,Outstanding credit value
4,2024-02-26,NaN,NaN,2.0,M3M500,M3 Mechanical Limited,YXS252215,38490,86.0,SL/Del,039-C-0692,NaN,NaN,SO0000413/1,NaN,SO0000413,Claimed,0.890474,5.3428,6,0,NaN,69.0,PRSCU,0.0,0.0,3.5762,1.7559,0.0,0.0,2.0,392.18,4.058900,9.25,3.003586,M3 MECHANICAL - YX,[D_InvoiceCost]*0.2490,01/01/0001,01/01/0001,U106,Supplier1,5,2024-02-26,2024-02-26,2024,2,5.3428,0.0,5.3428,0.0,5.3428,Derived from SupportData.xlsx,Outstanding,928,365+,True,Outstanding credit value


In [3]:
display(gl_account_summary_clean.head())


,gl_account_code,gl_account_description,gl_supplier_key,gl_entry_rows,gl_total_credit_amount,gl_first_entry_date,gl_last_entry_date,gl_current_balance,gl_current_balance_abs,supplier_group_code
0,01-85001,Supplier13 E071 Claimbacks,SUPPLIER13,105,242784.06,2026-05-01,2026-07-03,-242784.06,242784.06,SUPPLIER13
1,01-85002,Supplier2 E005 Claimbacks,SUPPLIER2,1,6240.00,2026-02-25,2026-02-25,-6240.00,6240.00,SUPPLIER2
2,01-85007,Supplier9 E022 Claimbacks,SUPPLIER9,5,5774.89,2026-02-09,2026-07-10,-5774.89,5774.89,SUPPLIER9
3,01-85017,Supplier3 E055 Claimbacks,SUPPLIER3,4,5550.00,2026-03-17,2026-07-14,-5550.00,5550.00,SUPPLIER3
4,01-85021,Supplier19 E070 Claimbacks,SUPPLIER19,4,9124.28,2026-04-20,2026-04-20,-9124.28,9124.28,SUPPLIER19


## Outstanding target

The GL credit-entry file provides stronger payment evidence than the support status field, but it is recorded at supplier/account level rather than claim-line level. For that reason, this section creates a supplier-level target where support owed is calculated for January to end-May 2026, while GL credit entries are included through June 2026 to reflect month-in-lieu payment timing. Suppliers with GL credit entries are treated as having payment evidence, while suppliers with support owed but no GL credit entries are treated as having no payment evidence.


In [4]:
claim_date_col = "claim_date" if "claim_date" in df.columns else "customer_date"
df[claim_date_col] = pd.to_datetime(df[claim_date_col], errors="coerce")
model_window = df[df[claim_date_col].between(analysis_start, analysis_support_end, inclusive="both")].copy()

model_window["supplier_group_code"] = model_window["Contract_Supplier_Code"].map(supplier_group_code)
model_window["credit_due_value"] = pd.to_numeric(model_window["credit_due_value"], errors="coerce").fillna(0)
model_window["days_since_claim"] = pd.to_numeric(model_window.get("days_since_claim", model_window.get("days_outstanding")), errors="coerce").fillna(-1)

source_type_mix = pd.crosstab(
    model_window["supplier_group_code"],
    model_window["SourceTransactionType"],
    values=model_window["credit_due_value"],
    aggfunc="sum",
).fillna(0)
source_type_mix.columns = [f"support_value_{str(col).replace('/', '_')}" for col in source_type_mix.columns]
source_type_mix = source_type_mix.reset_index()

supplier_support_summary = (
    model_window.groupby("supplier_group_code", dropna=False)
    .agg(
        support_line_rows=("credit_due_value", "size"),
        support_owed_value=("credit_due_value", "sum"),
        avg_claim_value=("credit_due_value", "mean"),
        max_claim_value=("credit_due_value", "max"),
        avg_days_since_claim=("days_since_claim", "mean"),
        first_claim_date=(claim_date_col, "min"),
        last_claim_date=(claim_date_col, "max"),
        supplier_name=("Contract_Supplier_Name", lambda s: "; ".join(sorted(set(s.dropna().astype(str))))),
    )
    .reset_index()
    .merge(source_type_mix, how="left", on="supplier_group_code")
)

supplier_gl_model_base = supplier_support_summary.merge(
    gl_account_summary_clean,
    how="left",
    on="supplier_group_code",
)

for col in ["gl_accounts", "gl_entry_rows", "gl_total_credit_amount", "gl_current_balance"]:
    if col in supplier_gl_model_base.columns:
        supplier_gl_model_base[col] = pd.to_numeric(supplier_gl_model_base[col], errors="coerce").fillna(0)

supplier_gl_model_base["gl_payment_coverage"] = np.where(
    supplier_gl_model_base["support_owed_value"].gt(0),
    supplier_gl_model_base["gl_total_credit_amount"] / supplier_gl_model_base["support_owed_value"],
    0,
)
supplier_gl_model_base["any_gl_payment_target"] = supplier_gl_model_base["gl_total_credit_amount"].gt(0).astype(int)
supplier_gl_model_base["gl_paid_in_full_proxy"] = supplier_gl_model_base["gl_payment_coverage"].ge(0.95).astype(int)
supplier_gl_model_base["gl_payment_status_proxy"] = np.select(
    [
        supplier_gl_model_base["gl_total_credit_amount"].le(0),
        supplier_gl_model_base["gl_payment_coverage"].lt(0.95),
    ],
    ["No GL payment evidence", "Part GL payment evidence"],
    default="GL paid in full/overpaid proxy",
)

gl_detail_cols = [col for col in [
    "supplier_group_code", "supplier_name", "support_line_rows", "support_owed_value",
    "gl_accounts", "gl_entry_rows", "gl_total_credit_amount", "gl_current_balance",
    "gl_payment_coverage", "gl_payment_status_proxy", "gl_account_codes", "gl_account_descriptions"
] if col in supplier_gl_model_base.columns]

print("Supplier-level GL payment target distribution:")
display(supplier_gl_model_base["any_gl_payment_target"].value_counts().rename(index={0: "No GL payment evidence", 1: "GL payment evidence"}))
display(supplier_gl_model_base[gl_detail_cols].sort_values("support_owed_value", ascending=False))


Supplier-level GL payment target distribution:


any_gl_payment_target
No GL payment evidence    23
Name: count, dtype: int64

,supplier_group_code,supplier_name,support_line_rows,support_owed_value,gl_entry_rows,gl_total_credit_amount,gl_current_balance,gl_payment_coverage,gl_payment_status_proxy
5,E071,Supplier13,702,248962.3200,0.0,0.0,0.0,0.0,No GL payment evidence
4,E070,Supplier19,421,106004.3636,0.0,0.0,0.0,0.0,No GL payment evidence
21,U102,Supplier4,220,88213.1200,0.0,0.0,0.0,0.0,No GL payment evidence
22,U106,Supplier1,9583,46163.5397,0.0,0.0,0.0,0.0,No GL payment evidence
10,E555,Supplier18,1299,41631.5931,0.0,0.0,0.0,0.0,No GL payment evidence
12,E907,Supplier21,106,35607.0000,0.0,0.0,0.0,0.0,No GL payment evidence
6,E102,Supplier7,438,9123.5200,0.0,0.0,0.0,0.0,No GL payment evidence
3,E059,Supplier5,13,7001.2000,0.0,0.0,0.0,0.0,No GL payment evidence
7,E175,Supplier8,1099,6630.9300,0.0,0.0,0.0,0.0,No GL payment evidence
17,L812,Supplier14,514,4705.8173,0.0,0.0,0.0,0.0,No GL payment evidence


## Supplier GL Payment Classifier

This classifier tests whether support-file features can separate suppliers with GL payment evidence from suppliers without GL payment evidence


RMSE is calculated against the predicted payment probability rather than a cash value. This means the error measures how far the model probability is from the actual GL evidence target of 0 or 1


In [5]:
gl_target_col = "any_gl_payment_target"
gl_leakage_cols = [
    "gl_accounts", "gl_entry_rows", "gl_total_credit_amount", "gl_current_balance",
    "gl_latest_running_balance", "gl_payment_coverage", "gl_paid_in_full_proxy",
    "gl_payment_status_proxy", "gl_account_codes", "gl_account_descriptions",
    "gl_first_entry_date", "gl_last_entry_date", gl_target_col,
]
gl_date_cols = supplier_gl_model_base.select_dtypes(include=["datetime64[ns]"]).columns.tolist()
gl_feature_cols = [
    col for col in supplier_gl_model_base.columns
    if col not in gl_leakage_cols + gl_date_cols
]

X_gl = supplier_gl_model_base[gl_feature_cols].copy()
y_gl = supplier_gl_model_base[gl_target_col].copy()

gl_numeric_features = X_gl.select_dtypes(include=["number", "bool"]).columns.tolist()
X_gl[gl_numeric_features] = X_gl[gl_numeric_features].astype(float)
gl_categorical_features = X_gl.select_dtypes(include=["object", "category", "string"]).columns.tolist()
X_gl[gl_categorical_features] = X_gl[gl_categorical_features].astype(str).fillna("missing")

print("GL classifier target classes:")
display(y_gl.value_counts())
print("GL classifier features:", gl_feature_cols)

if y_gl.nunique() < 2 or y_gl.value_counts().min() < 2 or len(y_gl) < 8:
    print("Not enough supplier-level GL target variation to train/test a classifier safely.")
    gl_model_results = pd.DataFrame()
else:
    X_gl_train, X_gl_test, y_gl_train, y_gl_test = train_test_split(
        X_gl,
        y_gl,
        test_size=0.35,
        random_state=random_state,
        stratify=y_gl,
    )

    gl_numeric_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ])
    gl_categorical_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
    gl_preprocess = ColumnTransformer(
        transformers=[
            ("num", gl_numeric_pipe, gl_numeric_features),
            ("cat", gl_categorical_pipe, gl_categorical_features),
        ],
        remainder="drop",
    )
    gl_classifier = Pipeline(steps=[
        ("preprocess", gl_preprocess),
        ("classifier", RandomForestClassifier(
            n_estimators=300,
            random_state=random_state,
            class_weight="balanced",
        )),
    ])

    gl_classifier.fit(X_gl_train, y_gl_train)
    gl_pred = gl_classifier.predict(X_gl_test)
    gl_pred_proba = gl_classifier.predict_proba(X_gl_test)[:, 1]

    print("GL-backed payment classifier evaluation")
    gl_probability_rmse = np.sqrt(np.mean((y_gl_test - gl_pred_proba) ** 2))
    print("Probability RMSE:", round(gl_probability_rmse, 3))
    print("Accuracy:", round((gl_pred == y_gl_test).mean(), 3))
    print(confusion_matrix(y_gl_test, gl_pred))
    print(classification_report(y_gl_test, gl_pred, target_names=["No GL payment evidence", "GL payment evidence"]))
    if y_gl_test.nunique() == 2:
        print("ROC AUC:", round(roc_auc_score(y_gl_test, gl_pred_proba), 3))

    gl_model_results = supplier_gl_model_base.loc[X_gl_test.index, ["supplier_group_code", "supplier_name", gl_target_col]].copy()
    gl_model_results["predicted_gl_payment_target"] = gl_pred
    gl_model_results["predicted_gl_payment_probability"] = gl_pred_proba
    display(gl_model_results.sort_values("predicted_gl_payment_probability", ascending=False))


GL classifier target classes:


any_gl_payment_target
0    23
Name: count, dtype: int64

GL classifier features: ['supplier_group_code', 'support_line_rows', 'support_owed_value', 'avg_claim_value', 'max_claim_value', 'avg_days_since_claim', 'supplier_name', 'support_value_SL_Crn', 'support_value_SL_Del', 'support_value_SL_Inv', 'gl_account_code', 'gl_account_description', 'gl_supplier_key', 'gl_current_balance_abs']
Not enough supplier-level GL target variation to train/test a classifier safely.


In [6]:
target_col = "outstanding_credit_value"

leakage_cols = [
    "supplier_reference_match_status",
    "supplier_reference_value",
    "credit_received_value",
    "credit_received_value_was_missing",
    "supplier_reference_value_was_missing",
    "target_unmatched_reference",
    "_merge",
    "requires_review",
    "high_value_review",
    "aged_review",
    "supplier_match_found",
    "credit_recovery_status",
    "claim_status",
    "is_claimed",
    "is_claimed_in_iq",
    "is_credit_owed_to_us",
    "unmatched_credit_value"
]

date_cols = df.select_dtypes(include=["datetime64[ns]"]).columns.tolist()

exclude_cols = leakage_cols + date_cols + [target_col] + ["credit_due_value"]

candidate_features = [
    col for col in df.columns
    if col not in exclude_cols
]

X = df[candidate_features].copy()
y = df[target_col].copy()

numeric_features = X.select_dtypes(include=["number", "bool"]).columns.tolist()

X[numeric_features] = X[numeric_features].astype(float)

categorical_features = X.select_dtypes(include=["object", "category", "string"]).columns.tolist()

print("Model target:", target_col)
print("Total features:", X.shape[1])
print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Example candidate features:", candidate_features[:20])


Model target: outstanding_credit_value
Total features: 47
Numeric features: 26
Categorical features: 21
Example candidate features: ['Month', 'Year', 'SalesDeliveryNote_Branch', 'Customer', 'Customer_Name', 'Product', 'Product_ManufacturerProductCode', 'Contract_Number', 'SourceTransactionType', 'Contract_D_ContractNumber', 'SalesInvoice_Number', 'SalesCreditNote_Number', 'SalesDeliveryNote_Number', 'SalesReturnNote_Number', 'SalesOrder_Number', 'Status', 'UnitClaimAmount', 'TotalClaimAmount', 'SalesDeliveryNoteLine_Quantity', 'SalesInvoiceLine_Quantity']


In [7]:
if y.nunique() < 2:
    raise ValueError("The target has only one unique value. Regression cannot be trained.")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=random_state
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)


Train shape: (54990, 47)
Test shape: (18330, 47)


In [8]:
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor

X[categorical_features] = X[categorical_features].astype(str).fillna("missing").replace("nan", "missing").replace("None", "missing")
X_train[categorical_features] = X_train[categorical_features].astype(str).fillna("missing").replace("nan", "missing").replace("None", "missing")
X_test[categorical_features] = X_test[categorical_features].astype(str).fillna("missing").replace("nan", "missing").replace("None", "missing")

numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent", fill_value="missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=25))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, numeric_features),
        ("cat", categorical_pipe, categorical_features)
    ],
    remainder="drop"
)

model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("regressor", RandomForestRegressor(
        n_estimators=50,
        max_depth=12,
        min_samples_leaf=10,
        random_state=random_state,
        n_jobs=-1
    ))
])

model.fit(X_train, y_train)

C:\Users\cathalhe\AppData\Roaming\Python\Python314\site-packages\sklearn\impute\_base.py:641: UserWarning: Skipping features without any observed values: ['Month' 'Year']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocess', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers 

In [9]:
from sklearn.metrics import mean_squared_error, r2_score

y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("Regression evaluation")
print("RMSE:", round(rmse, 2))
print("R2 score:", round(r2, 3))

model_results = pd.DataFrame({
    "actual_outstanding_credit": y_test,
    "predicted_outstanding_credit": y_pred,
})
model_results["prediction_error"] = model_results["actual_outstanding_credit"] - model_results["predicted_outstanding_credit"]
model_results["absolute_error"] = model_results["prediction_error"].abs()
display(model_results.describe())


C:\Users\cathalhe\AppData\Roaming\Python\Python314\site-packages\sklearn\impute\_base.py:641: UserWarning: Skipping features without any observed values: ['Month' 'Year']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


Regression evaluation
RMSE: 104.97
R2 score: 0.738


,actual_outstanding_credit,predicted_outstanding_credit,prediction_error,absolute_error
count,18330.000000,18330.000000,18330.000000,18330.000000
mean,35.410653,34.819883,0.590769,1.275972
std,205.049231,157.392673,104.975922,104.969829
min,-1582.500000,-594.783626,-987.716374,0.000000
25%,1.424175,1.422120,-0.012688,0.006550
50%,3.613500,3.618550,0.002239,0.015913
75%,11.690000,11.694050,0.018671,0.031649
max,18500.000000,4431.362451,14068.637549,14068.637549


In [10]:
def first_existing_model_col(df, candidates):
    for col in candidates:
        if col in df.columns:
            return col
    return None

scored = df.copy()
scored = scored.reset_index(drop=True)
if "days_since_claim" not in scored.columns and "days_outstanding" in scored.columns:
    scored["days_since_claim"] = scored["days_outstanding"]
scored["days_since_claim"] = pd.to_numeric(scored.get("days_since_claim"), errors="coerce").fillna(0)

scored["predicted_outstanding_credit"] = model.predict(X)
scored["predicted_outstanding_credit_rank"] = scored["predicted_outstanding_credit"].rank(pct=True)
scored["credit_due_value_rank"] = scored["credit_due_value"].rank(pct=True)
scored["days_since_claim_rank"] = scored["days_since_claim"].rank(pct=True)
scored["unmatched_risk"] = np.where(
    scored.get("supplier_reference_match_status", "").astype("string").eq("No supplier reference matched"),
    1,
    0
)
scored["unmatched_risk_rank"] = scored["unmatched_risk"].rank(pct=True)

scored["priority_score"] = (
    scored["predicted_outstanding_credit_rank"] * 0.45
    + scored["credit_due_value_rank"] * 0.35
    + scored["days_since_claim_rank"] * 0.15
    + scored["unmatched_risk_rank"] * 0.05
)

priority_list = scored.sort_values("priority_score", ascending=False)
supplier_display_col = first_existing_model_col(scored, [
    "Contract_Supplier_Name",
    "Contract_Supplier_Name_supplier",
    "Contract_Supplier_Name_customer",
    "Contract_Supplier"
])

priority_display_cols = [
    "Customer",
    supplier_display_col,
    "Product",
    "credit_due_value",
    "supplier_reference_value",
    "outstanding_credit_value",
    "days_since_claim",
    "supplier_reference_match_status",
    "predicted_outstanding_credit",
    "priority_score"
]
priority_display_cols = [col for col in priority_display_cols if col is not None and col in priority_list.columns]
display(priority_list[priority_display_cols].head(5))


C:\Users\cathalhe\AppData\Roaming\Python\Python314\site-packages\sklearn\impute\_base.py:641: UserWarning: Skipping features without any observed values: ['Month' 'Year']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


,Customer,Contract_Supplier_Name,Product,credit_due_value,supplier_reference_value,outstanding_credit_value,days_since_claim,supplier_reference_match_status,predicted_outstanding_credit,priority_score
419,KAN100GB,Supplier19,STEL148153,1298.3172,1298.3172,1298.3172,921,Derived from SupportData.xlsx,1303.443555,0.971964
871,CMK020,Supplier13,IDLOGIC24SV2,1500.0000,1500.0000,1500.0000,914,Derived from SupportData.xlsx,1511.484490,0.971744
418,KAN100GB,Supplier19,STEL7462208,1262.6145,1262.6145,1262.6145,921,Derived from SupportData.xlsx,1262.269824,0.971737
422,KAN100GB,Supplier19,STEL7462208,1262.6145,1262.6145,1262.6145,921,Derived from SupportData.xlsx,1262.269824,0.971737
601,KAN100GB,Supplier19,STEL7462208,1262.6145,1262.6145,1262.6145,919,Derived from SupportData.xlsx,1261.105046,0.971319


In [11]:
priority_display_cols = [
    "priority_score",
    "predicted_outstanding_credit",
    "credit_due_value",
    "supplier_reference_value",
    "outstanding_credit_value",
    "days_since_claim",
    "unmatched_risk",
    "Customer",
    "Customer_Name",
    "Product",
    "Product_ManufacturerProductCode",
    "Contract_Number",
    "Contract_D_ContractNumber",
    supplier_display_col,
    "Status",
    "supplier_reference_match_status"
]

available_priority_display_cols = [col for col in priority_display_cols if col is not None and col in priority_list.columns]
supplier_follow_up_priority_list = priority_list[available_priority_display_cols].copy()

display(supplier_follow_up_priority_list.head(5))


,priority_score,predicted_outstanding_credit,credit_due_value,supplier_reference_value,outstanding_credit_value,days_since_claim,unmatched_risk,Customer,Customer_Name,Product,Product_ManufacturerProductCode,Contract_Number,Contract_D_ContractNumber,Contract_Supplier_Name,Status,supplier_reference_match_status
419,0.971964,1303.443555,1298.3172,1298.3172,1298.3172,921,0,KAN100GB,Kane Group Building Services Ltd.,STEL148153,148153,1555.0,6147690,Supplier19,Claimed,Derived from SupportData.xlsx
871,0.971744,1511.484490,1500.0000,1500.0000,1500.0000,914,0,CMK020,C M K Mechanical Services Ltd.,IDLOGIC24SV2,228354,760.0,6137506,Supplier13,Claimed,Derived from SupportData.xlsx
418,0.971737,1262.269824,1262.6145,1262.6145,1262.6145,921,0,KAN100GB,Kane Group Building Services Ltd.,STEL7462208,7462208,1555.0,6147690,Supplier19,Claimed,Derived from SupportData.xlsx
422,0.971737,1262.269824,1262.6145,1262.6145,1262.6145,921,0,KAN100GB,Kane Group Building Services Ltd.,STEL7462208,7462208,1555.0,6147690,Supplier19,Claimed,Derived from SupportData.xlsx
601,0.971319,1261.105046,1262.6145,1262.6145,1262.6145,919,0,KAN100GB,Kane Group Building Services Ltd.,STEL7462208,7462208,1555.0,6147690,Supplier19,Claimed,Derived from SupportData.xlsx


In [12]:
for name in [
    "matched", "df_model_base", "quality_report", "missing_after_cleaning", "gl_account_summary",
    "gl_account_summary_clean", "supplier_follow_up_priority_list", "model_results",
    "classification_results", "regression_results", "valid_date_check", "summary",
    "year_view", "month_view", "supplier_summary", "monthly_summary"
]:
    if name in globals():
        obj = globals()[name]
        shape = getattr(obj, "shape", "")
        print(name, shape)
        try:
            display(obj.head(10))
        except AttributeError:
            display(obj)


gl_account_summary (15, 10)


,gl_account_code,gl_account_description,gl_supplier_key,gl_entry_rows,gl_total_credit_amount,gl_first_entry_date,gl_last_entry_date,gl_current_balance,gl_current_balance_abs,supplier_group_code
0,01-85001,Supplier13 E071 Claimbacks,SUPPLIER13,105,242784.06,2026-05-01,2026-07-03,-242784.06,242784.06,SUPPLIER13
1,01-85002,Supplier2 E005 Claimbacks,SUPPLIER2,1,6240.00,2026-02-25,2026-02-25,-6240.00,6240.00,SUPPLIER2
2,01-85007,Supplier9 E022 Claimbacks,SUPPLIER9,5,5774.89,2026-02-09,2026-07-10,-5774.89,5774.89,SUPPLIER9
3,01-85017,Supplier3 E055 Claimbacks,SUPPLIER3,4,5550.00,2026-03-17,2026-07-14,-5550.00,5550.00,SUPPLIER3
4,01-85021,Supplier19 E070 Claimbacks,SUPPLIER19,4,9124.28,2026-04-20,2026-04-20,-9124.28,9124.28,SUPPLIER19
5,01-85036,Supplier8 E175 Claimbacks,SUPPLIER8,2,4782.68,2026-04-23,2026-06-04,-4782.68,4782.68,SUPPLIER8
6,01-85046,Supplier10 E234 Claimbacks,SUPPLIER10,8,925.00,2026-04-08,2026-07-16,-925.00,925.00,SUPPLIER10
7,01-85056,Supplier17 E269A Claimbacks,SUPPLIER17,1,576.00,2026-06-17,2026-06-17,-576.00,576.00,SUPPLIER17
8,01-85079,Supplier18 E555 Claimbacks,SUPPLIER18,11,46476.48,2026-02-24,2026-07-22,-46476.48,46476.48,SUPPLIER18
9,01-85115,Supplier20 E948 Claimbacks,SUPPLIER20,5,3431.79,2026-02-24,2026-06-26,-3431.79,3431.79,SUPPLIER20


gl_account_summary_clean (15, 10)


,gl_account_code,gl_account_description,gl_supplier_key,gl_entry_rows,gl_total_credit_amount,gl_first_entry_date,gl_last_entry_date,gl_current_balance,gl_current_balance_abs,supplier_group_code
0,01-85001,Supplier13 E071 Claimbacks,SUPPLIER13,105,242784.06,2026-05-01,2026-07-03,-242784.06,242784.06,SUPPLIER13
1,01-85002,Supplier2 E005 Claimbacks,SUPPLIER2,1,6240.00,2026-02-25,2026-02-25,-6240.00,6240.00,SUPPLIER2
2,01-85007,Supplier9 E022 Claimbacks,SUPPLIER9,5,5774.89,2026-02-09,2026-07-10,-5774.89,5774.89,SUPPLIER9
3,01-85017,Supplier3 E055 Claimbacks,SUPPLIER3,4,5550.00,2026-03-17,2026-07-14,-5550.00,5550.00,SUPPLIER3
4,01-85021,Supplier19 E070 Claimbacks,SUPPLIER19,4,9124.28,2026-04-20,2026-04-20,-9124.28,9124.28,SUPPLIER19
5,01-85036,Supplier8 E175 Claimbacks,SUPPLIER8,2,4782.68,2026-04-23,2026-06-04,-4782.68,4782.68,SUPPLIER8
6,01-85046,Supplier10 E234 Claimbacks,SUPPLIER10,8,925.00,2026-04-08,2026-07-16,-925.00,925.00,SUPPLIER10
7,01-85056,Supplier17 E269A Claimbacks,SUPPLIER17,1,576.00,2026-06-17,2026-06-17,-576.00,576.00,SUPPLIER17
8,01-85079,Supplier18 E555 Claimbacks,SUPPLIER18,11,46476.48,2026-02-24,2026-07-22,-46476.48,46476.48,SUPPLIER18
9,01-85115,Supplier20 E948 Claimbacks,SUPPLIER20,5,3431.79,2026-02-24,2026-06-26,-3431.79,3431.79,SUPPLIER20


supplier_follow_up_priority_list (73320, 16)


,priority_score,predicted_outstanding_credit,credit_due_value,supplier_reference_value,outstanding_credit_value,days_since_claim,unmatched_risk,Customer,Customer_Name,Product,Product_ManufacturerProductCode,Contract_Number,Contract_D_ContractNumber,Contract_Supplier_Name,Status,supplier_reference_match_status
419,0.971964,1303.443555,1298.3172,1298.3172,1298.3172,921,0,KAN100GB,Kane Group Building Services Ltd.,STEL148153,148153,1555.0,6147690,Supplier19,Claimed,Derived from SupportData.xlsx
871,0.971744,1511.484490,1500.0000,1500.0000,1500.0000,914,0,CMK020,C M K Mechanical Services Ltd.,IDLOGIC24SV2,228354,760.0,6137506,Supplier13,Claimed,Derived from SupportData.xlsx
418,0.971737,1262.269824,1262.6145,1262.6145,1262.6145,921,0,KAN100GB,Kane Group Building Services Ltd.,STEL7462208,7462208,1555.0,6147690,Supplier19,Claimed,Derived from SupportData.xlsx
422,0.971737,1262.269824,1262.6145,1262.6145,1262.6145,921,0,KAN100GB,Kane Group Building Services Ltd.,STEL7462208,7462208,1555.0,6147690,Supplier19,Claimed,Derived from SupportData.xlsx
601,0.971319,1261.105046,1262.6145,1262.6145,1262.6145,919,0,KAN100GB,Kane Group Building Services Ltd.,STEL7462208,7462208,1555.0,6147690,Supplier19,Claimed,Derived from SupportData.xlsx
604,0.971319,1261.105046,1262.6145,1262.6145,1262.6145,919,0,KAN100GB,Kane Group Building Services Ltd.,STEL7462208,7462208,1555.0,6147690,Supplier19,Claimed,Derived from SupportData.xlsx
417,0.971073,1067.855222,1066.7844,1066.7844,1066.7844,921,0,KAN100GB,Kane Group Building Services Ltd.,STEL7462212,7462212,1555.0,6147690,Supplier19,Claimed,Derived from SupportData.xlsx
421,0.971073,1067.855222,1066.7844,1066.7844,1066.7844,921,0,KAN100GB,Kane Group Building Services Ltd.,STEL7462212,7462212,1555.0,6147690,Supplier19,Claimed,Derived from SupportData.xlsx
972,0.970834,1303.443555,1298.3172,1298.3172,1298.3172,912,0,KAN100GB,Kane Group Building Services Ltd.,STEL148153,148153,1555.0,6147690,Supplier19,Claimed,Derived from SupportData.xlsx
2058,0.970817,4431.362451,5940.0000,5940.0000,5940.0000,897,0,CMK020,C M K Mechanical Services Ltd.,IDLOGIC35V2,228309,1091.0,6137506,Supplier13,Claimed,Derived from SupportData.xlsx


model_results (18330, 4)


,actual_outstanding_credit,predicted_outstanding_credit,prediction_error,absolute_error
71483,82.3070,82.549387,-0.242387,0.242387
37749,13.6288,13.650638,-0.021838,0.021838
58171,160.0000,159.988495,0.011505,0.011505
15482,1.1563,1.150543,0.005757,0.005757
19922,8.9800,8.979934,0.000066,0.000066
42190,9.3289,9.304520,0.024380,0.024380
56952,7.1476,7.170994,-0.023394,0.023394
16187,1.6500,1.640515,0.009485,0.009485
14993,17.8600,17.907689,-0.047689,0.047689
32185,3.4095,3.438607,-0.029107,0.029107


In [13]:
for name in [
    "matched", "df_model_base", "quality_report", "missing_after_cleaning", "gl_account_summary",
    "gl_account_summary_clean", "supplier_follow_up_priority_list", "model_results",
    "classification_results", "regression_results", "valid_date_check", "summary",
    "year_view", "month_view", "supplier_summary", "monthly_summary"
]:
    if name in globals():
        obj = globals()[name]
        print(name, getattr(obj, "shape", ""))
        try:
            display(obj.head(10))
        except AttributeError:
            display(obj)


gl_account_summary (15, 10)


,gl_account_code,gl_account_description,gl_supplier_key,gl_entry_rows,gl_total_credit_amount,gl_first_entry_date,gl_last_entry_date,gl_current_balance,gl_current_balance_abs,supplier_group_code
0,01-85001,Supplier13 E071 Claimbacks,SUPPLIER13,105,242784.06,2026-05-01,2026-07-03,-242784.06,242784.06,SUPPLIER13
1,01-85002,Supplier2 E005 Claimbacks,SUPPLIER2,1,6240.00,2026-02-25,2026-02-25,-6240.00,6240.00,SUPPLIER2
2,01-85007,Supplier9 E022 Claimbacks,SUPPLIER9,5,5774.89,2026-02-09,2026-07-10,-5774.89,5774.89,SUPPLIER9
3,01-85017,Supplier3 E055 Claimbacks,SUPPLIER3,4,5550.00,2026-03-17,2026-07-14,-5550.00,5550.00,SUPPLIER3
4,01-85021,Supplier19 E070 Claimbacks,SUPPLIER19,4,9124.28,2026-04-20,2026-04-20,-9124.28,9124.28,SUPPLIER19
5,01-85036,Supplier8 E175 Claimbacks,SUPPLIER8,2,4782.68,2026-04-23,2026-06-04,-4782.68,4782.68,SUPPLIER8
6,01-85046,Supplier10 E234 Claimbacks,SUPPLIER10,8,925.00,2026-04-08,2026-07-16,-925.00,925.00,SUPPLIER10
7,01-85056,Supplier17 E269A Claimbacks,SUPPLIER17,1,576.00,2026-06-17,2026-06-17,-576.00,576.00,SUPPLIER17
8,01-85079,Supplier18 E555 Claimbacks,SUPPLIER18,11,46476.48,2026-02-24,2026-07-22,-46476.48,46476.48,SUPPLIER18
9,01-85115,Supplier20 E948 Claimbacks,SUPPLIER20,5,3431.79,2026-02-24,2026-06-26,-3431.79,3431.79,SUPPLIER20


gl_account_summary_clean (15, 10)


,gl_account_code,gl_account_description,gl_supplier_key,gl_entry_rows,gl_total_credit_amount,gl_first_entry_date,gl_last_entry_date,gl_current_balance,gl_current_balance_abs,supplier_group_code
0,01-85001,Supplier13 E071 Claimbacks,SUPPLIER13,105,242784.06,2026-05-01,2026-07-03,-242784.06,242784.06,SUPPLIER13
1,01-85002,Supplier2 E005 Claimbacks,SUPPLIER2,1,6240.00,2026-02-25,2026-02-25,-6240.00,6240.00,SUPPLIER2
2,01-85007,Supplier9 E022 Claimbacks,SUPPLIER9,5,5774.89,2026-02-09,2026-07-10,-5774.89,5774.89,SUPPLIER9
3,01-85017,Supplier3 E055 Claimbacks,SUPPLIER3,4,5550.00,2026-03-17,2026-07-14,-5550.00,5550.00,SUPPLIER3
4,01-85021,Supplier19 E070 Claimbacks,SUPPLIER19,4,9124.28,2026-04-20,2026-04-20,-9124.28,9124.28,SUPPLIER19
5,01-85036,Supplier8 E175 Claimbacks,SUPPLIER8,2,4782.68,2026-04-23,2026-06-04,-4782.68,4782.68,SUPPLIER8
6,01-85046,Supplier10 E234 Claimbacks,SUPPLIER10,8,925.00,2026-04-08,2026-07-16,-925.00,925.00,SUPPLIER10
7,01-85056,Supplier17 E269A Claimbacks,SUPPLIER17,1,576.00,2026-06-17,2026-06-17,-576.00,576.00,SUPPLIER17
8,01-85079,Supplier18 E555 Claimbacks,SUPPLIER18,11,46476.48,2026-02-24,2026-07-22,-46476.48,46476.48,SUPPLIER18
9,01-85115,Supplier20 E948 Claimbacks,SUPPLIER20,5,3431.79,2026-02-24,2026-06-26,-3431.79,3431.79,SUPPLIER20


supplier_follow_up_priority_list (73320, 16)


,priority_score,predicted_outstanding_credit,credit_due_value,supplier_reference_value,outstanding_credit_value,days_since_claim,unmatched_risk,Customer,Customer_Name,Product,Product_ManufacturerProductCode,Contract_Number,Contract_D_ContractNumber,Contract_Supplier_Name,Status,supplier_reference_match_status
419,0.971964,1303.443555,1298.3172,1298.3172,1298.3172,921,0,KAN100GB,Kane Group Building Services Ltd.,STEL148153,148153,1555.0,6147690,Supplier19,Claimed,Derived from SupportData.xlsx
871,0.971744,1511.484490,1500.0000,1500.0000,1500.0000,914,0,CMK020,C M K Mechanical Services Ltd.,IDLOGIC24SV2,228354,760.0,6137506,Supplier13,Claimed,Derived from SupportData.xlsx
418,0.971737,1262.269824,1262.6145,1262.6145,1262.6145,921,0,KAN100GB,Kane Group Building Services Ltd.,STEL7462208,7462208,1555.0,6147690,Supplier19,Claimed,Derived from SupportData.xlsx
422,0.971737,1262.269824,1262.6145,1262.6145,1262.6145,921,0,KAN100GB,Kane Group Building Services Ltd.,STEL7462208,7462208,1555.0,6147690,Supplier19,Claimed,Derived from SupportData.xlsx
601,0.971319,1261.105046,1262.6145,1262.6145,1262.6145,919,0,KAN100GB,Kane Group Building Services Ltd.,STEL7462208,7462208,1555.0,6147690,Supplier19,Claimed,Derived from SupportData.xlsx
604,0.971319,1261.105046,1262.6145,1262.6145,1262.6145,919,0,KAN100GB,Kane Group Building Services Ltd.,STEL7462208,7462208,1555.0,6147690,Supplier19,Claimed,Derived from SupportData.xlsx
417,0.971073,1067.855222,1066.7844,1066.7844,1066.7844,921,0,KAN100GB,Kane Group Building Services Ltd.,STEL7462212,7462212,1555.0,6147690,Supplier19,Claimed,Derived from SupportData.xlsx
421,0.971073,1067.855222,1066.7844,1066.7844,1066.7844,921,0,KAN100GB,Kane Group Building Services Ltd.,STEL7462212,7462212,1555.0,6147690,Supplier19,Claimed,Derived from SupportData.xlsx
972,0.970834,1303.443555,1298.3172,1298.3172,1298.3172,912,0,KAN100GB,Kane Group Building Services Ltd.,STEL148153,148153,1555.0,6147690,Supplier19,Claimed,Derived from SupportData.xlsx
2058,0.970817,4431.362451,5940.0000,5940.0000,5940.0000,897,0,CMK020,C M K Mechanical Services Ltd.,IDLOGIC35V2,228309,1091.0,6137506,Supplier13,Claimed,Derived from SupportData.xlsx


model_results (18330, 4)


,actual_outstanding_credit,predicted_outstanding_credit,prediction_error,absolute_error
71483,82.3070,82.549387,-0.242387,0.242387
37749,13.6288,13.650638,-0.021838,0.021838
58171,160.0000,159.988495,0.011505,0.011505
15482,1.1563,1.150543,0.005757,0.005757
19922,8.9800,8.979934,0.000066,0.000066
42190,9.3289,9.304520,0.024380,0.024380
56952,7.1476,7.170994,-0.023394,0.023394
16187,1.6500,1.640515,0.009485,0.009485
14993,17.8600,17.907689,-0.047689,0.047689
32185,3.4095,3.438607,-0.029107,0.029107


In [14]:
display(model_results.head())


,actual_outstanding_credit,predicted_outstanding_credit,prediction_error,absolute_error
71483,82.3070,82.549387,-0.242387,0.242387
37749,13.6288,13.650638,-0.021838,0.021838
58171,160.0000,159.988495,0.011505,0.011505
15482,1.1563,1.150543,0.005757,0.005757
19922,8.9800,8.979934,0.000066,0.000066
